## Step 1: Import Libraries

Import the libraries and configure interactive charts for Google Colab.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
import json, urllib.request, zipfile, io
from IPython.display import display, FileLink
pio.templates.default = 'plotly_white'
pio.renderers.default = 'colab' if 'google.colab' in __import__('sys').modules else 'plotly_mimetype'

## Step 2: Load the Dataset

Load the Online Retail dataset and display its dimensions and first five rows.

In [ ]:
source = Path('data/OnlineRetail.csv')
if not source.exists():
    url = 'https://www.kaggle.com/api/v1/datasets/download/vijayuv/onlineretail'
    with urllib.request.urlopen(url) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
        csv_name = next(n for n in archive.namelist() if n.lower().endswith('.csv'))
        source.parent.mkdir(parents=True, exist_ok=True)
        source.write_bytes(archive.read(csv_name))
raw = pd.read_csv(source, encoding='ISO-8859-1', dtype={'InvoiceNo':str,'StockCode':str})
print('Original shape:', raw.shape)
display(raw.head())

## Step 3: Inspect Data Quality

Inspect data types, missing values, duplicate rows and summary statistics.

In [ ]:
raw.info()
display(raw.isna().sum().rename('Missing values').to_frame())
print('Exact duplicate rows:', raw.duplicated().sum())
display(raw.describe())

## Step 4: Clean the Data

Remove incomplete, duplicate, cancelled and invalid sales records, then standardize the fields.

In [ ]:
df = raw.dropna(subset=['CustomerID']).drop_duplicates().copy()
df['Description'] = df['Description'].fillna('Unknown').str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
df = df.loc[~df['InvoiceNo'].str.startswith('C') & (df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
print('Cleaned rows:', len(df))
print('Excluded rows:', len(raw)-len(df))
assert not df.isna().any().any()
assert not df.duplicated().any()

## Step 5: Prepare Sales Measures

Calculate line amounts, create date and price categories, and summarize sales.

In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
df['PriceCategory'] = pd.cut(df['UnitPrice'], bins=[0,2,5,10,50,float('inf')], labels=['Very Low','Low','Medium','High','Premium'])
monthly = df.groupby('YearMonth',as_index=False).agg(Revenue=('TotalPrice','sum'),Orders=('InvoiceNo','nunique'))
country = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False)
print(f"Revenue: £{df.TotalPrice.sum():,.2f}")
print('Orders:', df.InvoiceNo.nunique(), 'Customers:', df.CustomerID.nunique())
display(df[['Quantity','UnitPrice','TotalPrice']].describe())

## Step 6: Bar Chart — International Markets

Compare revenue across the ten largest international markets, excluding the UK.

In [ ]:
international = country.drop('United Kingdom', errors='ignore').head(10).rename('Revenue').reset_index()
fig = px.bar(international, x='Country', y='Revenue', title='Top International Markets', labels={'Revenue':'Sales revenue (£)'})
fig.show()

## Step 7: Line Chart — Monthly Revenue

Show monthly revenue trends, noting that December 2011 is incomplete.

In [ ]:
fig = px.line(monthly, x='YearMonth', y='Revenue', markers=True, title='Monthly Sales Revenue', labels={'YearMonth':'Month','Revenue':'Sales revenue (£)'})
fig.show()

## Step 8: Histogram — Transaction Quantities

Show the frequency of transaction quantities from 1 to 100 units.

In [ ]:
fig = px.histogram(df.loc[df.Quantity.le(100)], x='Quantity', title='Quantity per Transaction Line (1–100 Units)', labels={'Quantity':'Units on one transaction line'})
fig.update_traces(xbins=dict(start=0,end=100,size=2), autobinx=False)
fig.update_layout(xaxis_range=[0,100], yaxis_title='Transaction lines')
fig.show()

## Step 9: Pie Chart — Country Revenue Share

Show revenue shares for the five largest markets and group the rest as Others.

In [ ]:
pie = country.head(5).copy()
pie.loc['Others'] = country.iloc[5:].sum()
fig = px.pie(names=pie.index, values=pie.values, title='Sales Revenue by Country')
fig.update_traces(textinfo='percent+label', hovertemplate='%{label}: £%{value:,.2f}<extra></extra>')
fig.show()

## Step 10: Scatter Plot — Quantity and Line Amount

Explore quantity and line amount using up to 5,000 sampled records on logarithmic axes.

In [ ]:
sample = df.sample(min(5000,len(df)),random_state=42)
fig = px.scatter(sample,x='Quantity',y='TotalPrice',color='PriceCategory',log_x=True,log_y=True,hover_data=['Description','Country'],title='Quantity and Line Amount (5,000-row Sample)',labels={'TotalPrice':'Line amount (£)'})
fig.show()

## Step 11: Box Plot — Line Amount by Price Category

Compare line amounts across price categories using the same sample and a logarithmic scale.

In [ ]:
fig = px.box(sample,x='PriceCategory',y='TotalPrice',color='PriceCategory',log_y=True,points='outliers',category_orders={'PriceCategory':['Very Low','Low','Medium','High','Premium']},title='Line Amount by Price Category (5,000-row Sample)',labels={'TotalPrice':'Line amount (£)'})
fig.show()

## Step 12: Heatmap — Weekday and Hour

Compare total revenue by weekday and hour using colour intensity.

In [ ]:
days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat = df.pivot_table(index='DayOfWeek',columns='Hour',values='TotalPrice',aggfunc='sum',fill_value=0).reindex(index=days,columns=range(24),fill_value=0)
fig = px.imshow(heat,aspect='auto',color_continuous_scale='Teal',title='Sales Revenue by Weekday and Hour',labels={'x':'Hour','y':'Weekday','color':'Revenue (£)'})
fig.show()

## Step 13: Grouped Bar Chart — Germany and France

Compare monthly revenue for Germany and France with grouped bars.

In [ ]:
grouped = df[df.Country.isin(['Germany','France'])].groupby(['YearMonth','Country'],as_index=False)['TotalPrice'].sum()
fig = px.bar(grouped,x='YearMonth',y='TotalPrice',color='Country',barmode='group',title='Germany and France: Monthly Sales',labels={'YearMonth':'Month','TotalPrice':'Sales revenue (£)'})
fig.show()

## Step 14: Summarize Findings

Summarize the UK revenue share, peak sales periods, and country and product counts.

In [ ]:
print(f"UK share of sales revenue: {country.get('United Kingdom',0)/df.TotalPrice.sum():.1%}")
peak = monthly.loc[monthly.Revenue.idxmax()]
print(f"Highest-revenue month: {peak.YearMonth}, £{peak.Revenue:,.2f}")
print('Highest-revenue invoice hour:', df.groupby('Hour').TotalPrice.sum().idxmax())
print('Countries:', df.Country.nunique(), 'Products:', df.StockCode.nunique())

## Step 15: Export Data for the Dashboard

Export the cleaned CSV and dashboard data file, then display their download links.

In [ ]:
df.to_csv('final_dataset.csv',index=False)
display(FileLink('final_dataset.csv'))
dictionaries = {c:sorted(df[c].astype(str).unique().tolist()) for c in ['StockCode','Description','Country','PriceCategory']}
encoded = {c:df[c].astype(str).map({v:i for i,v in enumerate(values)}).tolist() for c,values in dictionaries.items()}
rows=[]
for i,r in enumerate(df.itertuples()):
    rows.append([r.InvoiceNo,encoded['StockCode'][i],encoded['Description'][i],float(r.Quantity),r.InvoiceDate.strftime('%Y-%m-%dT%H:%M:%S'),float(r.UnitPrice),r.CustomerID,encoded['Country'][i],encoded['PriceCategory'][i]])
summary={'raw_rows':len(raw),'cleaned_rows':len(df),'revenue':round(float(df.TotalPrice.sum()),2),'orders':int(df.InvoiceNo.nunique()),'customers':int(df.CustomerID.nunique()),'start':str(df.InvoiceDate.min()),'end':str(df.InvoiceDate.max())}
payload={'dictionaries':dictionaries,'rows':rows,'summary':summary}
Path('retail-data.js').write_text('window.RETAIL='+json.dumps(payload,separators=(',',':')).replace('</','<\/')+';')
display(FileLink('retail-data.js'))
